<a href="https://colab.research.google.com/github/tanakaryotadayo-wq/-/blob/jules-12126073915346507137-2b559607/Semantic_Delta_Engine_with_AST_Enrichment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
Semantic Delta Engine — パス非依存の意味的差分追跡 ＋ ASTベクトル濃縮

5つの要件を実装:
  1. Canonical Text Embedding (パス除去テキスト)
  2. Content Hash <-> Path 分離 (content_index テーブル)
  3. Delta Vector 保存 (差分ベクトルそのもの)
  4. Model Fingerprint 固定 (再埋め込み判定)
  5. [NEW] AST Vector Enrichment (グラフ畳み込みによる構造と意味の融合)

依存: Python 3 stdlib のみ
"""

from __future__ import annotations

import hashlib
import json
import math
import os
import re
import sqlite3
import sys
import time
from contextlib import contextmanager
from datetime import datetime, timezone
from typing import Dict, List, Optional, Sequence, Tuple

# --- Constants ---

VECTOR_DIM = 3072
DB_PATH = os.environ.get("EMBEDDING2_DB_PATH", os.path.join(os.path.dirname(__file__), "neural_packets.db"))

# パスと判定されるパターン (Unix / Windows / URL)
_PATH_PATTERN = re.compile(
    r"""(?:                       # non-capturing group
        /[\w.@:~-]+(?:/[\w.@:~-]+)+   # Unix absolute: /foo/bar
      | [A-Za-z]:\\[\w.~-]+(?:\\[\w.~-]+)+  # Windows: C:\foo\bar
      | [\w.@-]+(?:/[\w.@-]+){2,}     # relative multi-segment: a/b/c
    )""",
    re.VERBOSE,
)

# モデル名の正規マップ (mode -> fingerprint)
MODEL_FINGERPRINTS: Dict[str, str] = {
    "offline": "sha256-pseudo-3072d",
    "gemini": "gemini-embedding-2-preview",
    "local_ai": "bge-m3-mlx-fp16",
}


# --- Pure functions ---

def _strip_paths(text: str) -> str:
    """テキストからファイルパス風の部分を除去する。"""
    return _PATH_PATTERN.sub(
        lambda match: match.group(0).replace("\\", "/").rsplit("/", 1)[-1],
        text,
    )

def _tail_token(path_like: str) -> str:
    """パスからファイル名 (末尾トークン) だけを返す。"""
    normalized = path_like.replace("\\", "/")
    return normalized.rsplit("/", 1)[-1] if "/" in normalized else normalized

def _normalize_packet_value(value: object, *, strip_paths: bool = False, tail: bool = False) -> str:
    if value is None:
        return ""
    if isinstance(value, bool):
        text = "true" if value else "false"
    else:
        text = str(value)
    if tail:
        text = _tail_token(text)
    if strip_paths:
        text = _strip_paths(text)
    return " ".join(text.split())

def _append_packet_field(parts: List[str], label: str, value: object, *, strip_paths: bool = False, tail: bool = False) -> None:
    normalized = _normalize_packet_value(value, strip_paths=strip_paths, tail=tail)
    if normalized:
        parts.append(f"{label}:{normalized}")

def _append_packet_list(parts: List[str], label: str, values: object, *, strip_paths: bool = False, tail: bool = False) -> None:
    if not isinstance(values, list):
        return
    for value in values:
        _append_packet_field(parts, label, value, strip_paths=strip_paths, tail=tail)

def _append_evidence(parts: List[str], evidence: object) -> None:
    if not isinstance(evidence, list):
        return
    for item in evidence:
        if not isinstance(item, dict):
            continue
        path = _normalize_packet_value(item.get("path"), tail=True)
        lines = _normalize_packet_value(item.get("lines"))
        text = f"{path}:{lines}" if path and lines else path or lines
        if text:
            parts.append(f"evidence:{text}")

def _build_source_doc_id(source_uri: str) -> str:
    digest = hashlib.sha1(source_uri.encode("utf-8")).hexdigest()[:16]
    return f"doc:{digest}"

def _coerce_source_doc_uri(packet: dict) -> str:
    explicit = packet.get("source_doc_uri")
    if explicit:
        return str(explicit)
    repo = str(packet.get("repo", "") or "")
    ref = str(packet.get("ref", "") or "")
    skill = packet.get("skill") or {}
    code_ref = str(skill.get("code_ref", "") or "")

    if repo and ref: return f"{repo}@{ref}"
    if repo and code_ref: return f"{repo}::{code_ref}"
    if ref: return ref
    if code_ref: return code_ref
    return str(packet.get("id", "unknown"))

def _coerce_source_doc_id(packet: dict) -> str:
    explicit = packet.get("source_doc_id") or packet.get("_source_doc_id")
    if explicit: return str(explicit)
    return _build_source_doc_id(_coerce_source_doc_uri(packet))

def _coerce_source_doc_title(packet: dict) -> str:
    explicit = packet.get("source_doc_title")
    if explicit: return str(explicit)
    skill = packet.get("skill") or {}
    for candidate in (skill.get("code_ref"), packet.get("ref"), packet.get("id")):
        title = _normalize_packet_value(candidate, tail=True)
        if title: return title
    return "canonical-document"

def _coerce_source_doc_metadata(packet: dict) -> dict:
    skill = packet.get("skill") or {}
    return {
        "repo": str(packet.get("repo", "") or ""),
        "ref": str(packet.get("ref", "") or ""),
        "code_ref": str(skill.get("code_ref", "") or ""),
        "packet_id": str(packet.get("id", "") or ""),
        "verifier_log_ref": str(packet.get("verifier_log_ref", "") or ""),
    }

def canonical_text(packet: dict) -> str:
    """[要件1] パス情報を除去した正規テキストを生成する。"""
    trigger = packet.get("trigger") or {}
    skill = packet.get("skill") or {}
    exec_profile = packet.get("exec_profile") or {}
    verifier = packet.get("verifier") or {}
    kv = packet.get("kv") or {}

    raw_id = str(packet.get("id", ""))
    id_token = _tail_token(raw_id) if "/" in raw_id or "\\" in raw_id else raw_id

    parts: List[str] = []
    _append_packet_field(parts, "id", id_token)
    _append_packet_field(parts, "status", packet.get("status", ""))
    _append_packet_field(parts, "fail_reason", packet.get("fail_reason", ""))
    _append_packet_field(parts, "license", packet.get("license", ""))
    _append_packet_field(parts, "toolchain", packet.get("toolchain_fingerprint", ""), strip_paths=True)
    _append_packet_field(parts, "deps_lock", packet.get("deps_lock_ref", ""), tail=True)
    _append_packet_field(parts, "model", packet.get("model_fingerprint", ""))
    _append_packet_field(parts, "note", packet.get("notes", ""), strip_paths=True)
    _append_packet_list(parts, "concept", trigger.get("concepts") or [])
    _append_packet_field(parts, "language", skill.get("language", ""))
    _append_packet_field(parts, "input", skill.get("input_spec", ""))
    _append_packet_field(parts, "output", skill.get("output_spec", ""))
    _append_packet_list(parts, "dependency", skill.get("dependencies") or [])
    _append_packet_field(parts, "code", skill.get("code_ref", ""), tail=True)
    _append_packet_field(parts, "exec_mode", exec_profile.get("mode"))
    _append_packet_field(parts, "timeout_sec", exec_profile.get("timeout_sec"))
    _append_packet_field(parts, "memory_mb", exec_profile.get("memory_mb"))
    _append_packet_field(parts, "cpus", exec_profile.get("cpus"))
    _append_packet_field(parts, "network", exec_profile.get("network"))
    _append_packet_field(parts, "fs", exec_profile.get("fs"))
    _append_packet_field(parts, "verifier_level", verifier.get("level", ""))
    _append_packet_field(parts, "verifier_type", verifier.get("type", ""))
    _append_packet_field(parts, "pass_condition", verifier.get("pass_condition", ""))
    _append_packet_field(parts, "verifier_cmd", verifier.get("cmd", ""), strip_paths=True)
    _append_packet_field(parts, "verifier_log", packet.get("verifier_log_ref", ""), tail=True)
    _append_evidence(parts, packet.get("evidence"))
    _append_packet_field(parts, "kv_eligible", kv.get("eligible"))
    _append_packet_field(parts, "prefix", kv.get("canonical_prefix_id", ""), tail=True)
    _append_packet_field(parts, "kv_ref", kv.get("kv_ref", ""), tail=True)
    return " ".join(parts)

def content_hash(text: str) -> str:
    """[要件2] テキストの SHA-256 ハッシュを返す。"""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def compute_delta(v_before: Sequence[float], v_after: Sequence[float]) -> Tuple[List[float], float]:
    """[要件3] 差分ベクトルとドリフトスコアを計算する。"""
    if len(v_before) != len(v_after):
        raise ValueError(f"Vector dimensionality mismatch: {len(v_before)} vs {len(v_after)}")
    delta = [a - b for a, b in zip(v_after, v_before)]

    dot = sum(a * b for a, b in zip(v_before, v_after))
    mag_a = math.sqrt(sum(x * x for x in v_before))
    mag_b = math.sqrt(sum(x * x for x in v_after))
    cos_sim = 0.0 if mag_a == 0.0 or mag_b == 0.0 else dot / (mag_a * mag_b)

    drift = 1.0 - cos_sim
    return delta, drift

def reconstruct(base_vector: Sequence[float], delta: Sequence[float]) -> List[float]:
    """[要件3] base + delta で元のベクトルを完全復元する。"""
    return [b + d for b, d in zip(base_vector, delta)]

def model_fingerprint(mode: str) -> str:
    """[要件4] モード名からモデルフィンガープリントを解決する。"""
    return MODEL_FINGERPRINTS.get(mode, f"unknown:{mode}")


# --- Engine class ---

class SemanticDeltaEngine:
    """パス非依存の意味的差分追跡 ＋ ASTベクトル濃縮エンジン"""

    def __init__(self, db_path: Optional[str] = None):
        self.db_path = db_path or DB_PATH
        self._ensure_schema()

    def _connect(self) -> sqlite3.Connection:
        conn = sqlite3.connect(self.db_path, uri=self.db_path.startswith("file:"))
        conn.row_factory = sqlite3.Row
        conn.execute("PRAGMA busy_timeout = 5000")
        return conn

    @contextmanager
    def _connection(self):
        conn = self._connect()
        try:
            yield conn
            conn.commit()
        except Exception:
            try: conn.rollback()
            except sqlite3.Error: pass
            raise
        finally:
            conn.close()

    @staticmethod
    def _ensure_column(conn: sqlite3.Connection, table_name: str, column_name: str, column_definition: str) -> None:
        columns = {row["name"] for row in conn.execute(f"PRAGMA table_info({table_name})").fetchall()}
        if column_name not in columns:
            conn.execute(f"ALTER TABLE {table_name} ADD COLUMN {column_name} {column_definition}")

    def _ensure_schema(self) -> None:
        with self._connection() as conn:
            conn.execute(
                """CREATE TABLE IF NOT EXISTS canonical_documents (
                    source_doc_id TEXT PRIMARY KEY,
                    source_uri TEXT NOT NULL,
                    title TEXT DEFAULT '',
                    document_hash TEXT DEFAULT '',
                    metadata TEXT DEFAULT '{}',
                    created_at TEXT NOT NULL,
                    updated_at TEXT NOT NULL
                )"""
            )
            # [要件2]
            conn.execute(
                """CREATE TABLE IF NOT EXISTS content_index (
                    content_hash  TEXT PRIMARY KEY,
                    paths         TEXT NOT NULL DEFAULT '[]',
                    packet_id     TEXT,
                    source_doc_id TEXT,
                    canonical_text TEXT NOT NULL,
                    updated_at    TEXT NOT NULL
                )"""
            )
            # [要件3]
            conn.execute(
                """CREATE TABLE IF NOT EXISTS delta_vectors (
                    id               INTEGER PRIMARY KEY AUTOINCREMENT,
                    packet_id        TEXT NOT NULL,
                    base_vector_hash TEXT NOT NULL,
                    delta_vector     TEXT NOT NULL,
                    drift_score      REAL NOT NULL,
                    model_fingerprint TEXT NOT NULL,
                    drift_type       TEXT DEFAULT 'semantic', -- 'semantic' or 'structural'
                    created_at       TEXT NOT NULL
                )"""
            )

            # [要件5: NEW] ASTの依存関係（エッジ）を保存するテーブル
            conn.execute(
                """CREATE TABLE IF NOT EXISTS packet_edges (
                    source_packet_id TEXT NOT NULL,
                    target_packet_id TEXT NOT NULL,
                    edge_type TEXT DEFAULT 'dependency',
                    PRIMARY KEY (source_packet_id, target_packet_id)
                )"""
            )

            conn.execute("CREATE INDEX IF NOT EXISTS idx_delta_packet ON delta_vectors(packet_id)")
            conn.execute("CREATE INDEX IF NOT EXISTS idx_content_index_source_doc ON content_index(source_doc_id)")
            self._ensure_column(conn, "content_index", "source_doc_id", "TEXT")
            self._ensure_column(conn, "delta_vectors", "drift_type", "TEXT DEFAULT 'semantic'")

            try:
                self._ensure_column(conn, "packets", "embedding_model", "TEXT")
                self._ensure_column(conn, "packets", "source_doc_id", "TEXT")
            except sqlite3.OperationalError:
                pass

    # ---- 要件1 & 2: Content Management ----

    def canonical_text(self, packet: dict) -> str:
        return canonical_text(packet)

    def register_content(self, packet: dict) -> str:
        text = canonical_text(packet)
        c_hash = content_hash(text)
        packet_id = str(packet.get("id", ""))
        now = datetime.now(tz=timezone.utc).isoformat()

        # [NEW] 依存関係も抽出してエッジテーブルに登録
        dependencies = packet.get("skill", {}).get("dependencies") or []

        last_exc: Optional[sqlite3.OperationalError] = None
        for attempt in range(10):
            try:
                with self._connection() as conn:
                    conn.execute("BEGIN IMMEDIATE")
                    source_doc_id = self._register_canonical_document_with_conn(conn, packet, text, c_hash)

                    conn.execute(
                        """INSERT OR IGNORE INTO content_index
                           (content_hash, paths, packet_id, source_doc_id, canonical_text, updated_at)
                           VALUES (?, ?, ?, ?, ?, ?)""",
                        (c_hash, json.dumps([packet_id], ensure_ascii=False), packet_id, source_doc_id, text, now),
                    )
                    existing = conn.execute("SELECT paths, source_doc_id FROM content_index WHERE content_hash = ?", (c_hash,)).fetchone()
                    if existing is None: raise RuntimeError(f"content_index missing row after insert: {c_hash}")

                    paths = json.loads(existing["paths"])
                    if packet_id not in paths: paths.append(packet_id)

                    conn.execute(
                        """UPDATE content_index
                           SET paths = ?, packet_id = ?, source_doc_id = ?, canonical_text = ?, updated_at = ?
                           WHERE content_hash = ?""",
                        (json.dumps(paths, ensure_ascii=False), packet_id, existing["source_doc_id"] or source_doc_id, text, now, c_hash),
                    )

                    # 依存関係の登録 (AST Topology)
                    for dep_id in dependencies:
                        conn.execute(
                            """INSERT OR IGNORE INTO packet_edges (source_packet_id, target_packet_id)
                               VALUES (?, ?)""",
                            (packet_id, dep_id)
                        )

                    try:
                        conn.execute("UPDATE packets SET source_doc_id = ? WHERE id = ?", (source_doc_id, packet_id))
                    except sqlite3.OperationalError: pass
                return c_hash
            except sqlite3.OperationalError as exc:
                last_exc = exc
                if "locked" not in str(exc).lower(): raise
                time.sleep(0.01 * (attempt + 1))
        raise last_exc or sqlite3.OperationalError("database locked")

    def _register_canonical_document_with_conn(self, conn: sqlite3.Connection, packet: dict, text: str, c_hash: str) -> str:
        source_doc_id = _coerce_source_doc_id(packet)
        source_uri = _coerce_source_doc_uri(packet)
        title = _coerce_source_doc_title(packet)
        metadata = _coerce_source_doc_metadata(packet)
        now = datetime.now(tz=timezone.utc).isoformat()
        conn.execute(
            """INSERT OR IGNORE INTO canonical_documents (source_doc_id, source_uri, title, document_hash, metadata, created_at, updated_at)
               VALUES (?, ?, ?, ?, ?, ?, ?)""",
            (source_doc_id, source_uri, title, c_hash, json.dumps(metadata, ensure_ascii=False, sort_keys=True), now, now)
        )
        conn.execute(
            """UPDATE canonical_documents SET source_uri = ?, title = ?, document_hash = ?, metadata = ?, updated_at = ? WHERE source_doc_id = ?""",
            (source_uri, title, c_hash, json.dumps({**metadata, "canonical_text_preview": text[:200]}, ensure_ascii=False, sort_keys=True), now, source_doc_id)
        )
        return source_doc_id

    def resolve_by_hash(self, c_hash: str) -> Optional[dict]:
        with self._connection() as conn:
            row = conn.execute(
                """SELECT ci.*, cd.source_uri, cd.title, cd.document_hash, cd.metadata AS source_metadata
                   FROM content_index ci LEFT JOIN canonical_documents cd ON cd.source_doc_id = ci.source_doc_id
                   WHERE ci.content_hash = ?""", (c_hash,)
            ).fetchone()
            if not row: return None
            return {
                "content_hash": row["content_hash"],
                "paths": json.loads(row["paths"]),
                "packet_id": row["packet_id"],
                "source_doc_id": row["source_doc_id"],
                "canonical_text": row["canonical_text"],
                "updated_at": row["updated_at"],
            }

    # ---- 要件5: AST Vector Enrichment (NEW) ----

    def enrich_vector(self, packet_id: str, current_vector: Sequence[float], alpha: float = 0.85) -> List[float]:
        """
        [NEW] グラフ畳み込みを利用し、依存先ノードのベクトルを自身のベクトルに濃縮する。
        H^(k) = alpha * H_self + (1-alpha) * Mean(H_neighbors)

        alpha: 自己ベクトル(テキスト本来の意味)の保持率。
        """
        with self._connection() as conn:
            # 自分が依存している対象 (Children/Dependencies) を取得
            edges = conn.execute(
                "SELECT target_packet_id FROM packet_edges WHERE source_packet_id = ?",
                (packet_id,)
            ).fetchall()

            neighbor_ids = [row["target_packet_id"] for row in edges]
            if not neighbor_ids:
                return list(current_vector) # 依存関係がなければそのまま

            # neighbor_ids の現在のベクトルを取得 (packets テーブルから)
            neighbor_vectors = []
            for nid in neighbor_ids:
                try:
                    row = conn.execute("SELECT vector FROM packets WHERE id = ?", (nid,)).fetchone()
                    if row and row["vector"]:
                        neighbor_vectors.append(json.loads(row["vector"]))
                except sqlite3.OperationalError:
                    pass # packets テーブルがない場合は無視

            if not neighbor_vectors:
                return list(current_vector)

            # Mean(H_neighbors) の計算
            dim = len(current_vector)
            mean_neighbor = [0.0] * dim
            for nv in neighbor_vectors:
                for i in range(dim):
                    mean_neighbor[i] += nv[i]
            mean_neighbor = [val / len(neighbor_vectors) for val in mean_neighbor]

            # 濃縮（融合）: 自己情報(alpha) + 周辺情報(1-alpha)
            enriched = [
                alpha * current_vector[i] + (1.0 - alpha) * mean_neighbor[i]
                for i in range(dim)
            ]

            return enriched

    # ---- 要件3: Delta Vectors ----

    def store_delta(
        self,
        packet_id: str,
        base_vector: Sequence[float],
        current_vector: Sequence[float],
        mode: str = "offline",
        drift_type: str = "semantic"
    ) -> dict:
        """
        差分ベクトルを計算して保存する。
        AST Enrichmentで生じた差分を保存する場合は drift_type="structural" を指定する。
        """
        delta, drift = compute_delta(base_vector, current_vector)
        base_hash = hashlib.sha256(json.dumps(list(base_vector), separators=(",", ":")).encode()).hexdigest()[:16]
        fp = model_fingerprint(mode)
        now = datetime.now(tz=timezone.utc).isoformat()

        with self._connection() as conn:
            cur = conn.execute(
                """INSERT INTO delta_vectors
                   (packet_id, base_vector_hash, delta_vector, drift_score, model_fingerprint, drift_type, created_at)
                   VALUES (?, ?, ?, ?, ?, ?, ?)""",
                (packet_id, base_hash, json.dumps(delta, separators=(",", ":")), drift, fp, drift_type, now),
            )
            return {
                "delta_id": cur.lastrowid,
                "drift_score": drift,
                "model_fingerprint": fp,
                "drift_type": drift_type
            }

    def get_deltas(self, packet_id: str) -> List[dict]:
        with self._connection() as conn:
            rows = conn.execute(
                """SELECT id, packet_id, base_vector_hash, drift_score, model_fingerprint, drift_type, created_at
                   FROM delta_vectors WHERE packet_id = ? ORDER BY created_at""",
                (packet_id,)
            ).fetchall()
            return [dict(r) for r in rows]

    def get_delta_vector(self, delta_id: int) -> Optional[List[float]]:
        with self._connection() as conn:
            row = conn.execute("SELECT delta_vector FROM delta_vectors WHERE id = ?", (delta_id,)).fetchone()
            if not row: return None
            return json.loads(row["delta_vector"])

    def stats(self) -> dict:
        with self._connection() as conn:
            canonical_doc_count = conn.execute("SELECT COUNT(*) FROM canonical_documents").fetchone()[0]
            ci_count = conn.execute("SELECT COUNT(*) FROM content_index").fetchone()[0]
            dv_count = conn.execute("SELECT COUNT(*) FROM delta_vectors").fetchone()[0]
            edge_count = conn.execute("SELECT COUNT(*) FROM packet_edges").fetchone()[0]
            avg_drift = conn.execute("SELECT AVG(drift_score) FROM delta_vectors").fetchone()[0] or 0.0
            max_drift = conn.execute("SELECT MAX(drift_score) FROM delta_vectors").fetchone()[0] or 0.0

            models = {r["model_fingerprint"]: r["cnt"] for r in conn.execute(
                "SELECT model_fingerprint, COUNT(*) as cnt FROM delta_vectors GROUP BY model_fingerprint"
            ).fetchall()}

        return {
            "canonical_documents": canonical_doc_count,
            "content_index_entries": ci_count,
            "packet_edges (AST)": edge_count,
            "delta_vectors_stored": dv_count,
            "avg_drift": round(avg_drift, 6),
            "max_drift": round(max_drift, 6),
            "models_used": models,
        }

# --- CLI & Self Test ---

def _run_self_test() -> int:
    print("Semantic Delta Engine + AST Enrichment Self-Test")
    print("=" * 50)
    passed, total = 0, 0

    def check(name: str, condition: bool) -> None:
        nonlocal passed, total
        total += 1
        if condition:
            passed += 1
            print(f"  ✅ {name}")
        else:
            print(f"  ❌ {name}")

    # 1. Packet preparation with dependencies
    pkt_parent = {
        "id": "parent.py",
        "skill": {"dependencies": ["child.py"]}
    }
    pkt_child = {
        "id": "child.py",
        "skill": {"dependencies": []}
    }

    uri = "file:semantic_delta_selftest?mode=memory&cache=shared"
    anchor = sqlite3.connect(uri, uri=True)
    try:
        engine = SemanticDeltaEngine(db_path=uri)
        # Setup dummy packets table
        anchor.execute("CREATE TABLE packets (id TEXT PRIMARY KEY, vector TEXT)")

        # 1. Register Contents (extracts edges)
        engine.register_content(pkt_child)
        engine.register_content(pkt_parent)

        edges = anchor.execute("SELECT * FROM packet_edges").fetchall()
        check("AST Edges properly extracted from packet", len(edges) == 1 and edges[0][0] == "parent.py" and edges[0][1] == "child.py")

        # 2. Insert Base Vectors
        v_parent = [1.0, 0.0, 0.0]
        v_child = [0.0, 1.0, 0.0]
        anchor.execute("INSERT INTO packets (id, vector) VALUES (?, ?)", ("parent.py", json.dumps(v_parent)))
        anchor.execute("INSERT INTO packets (id, vector) VALUES (?, ?)", ("child.py", json.dumps(v_child)))
        anchor.commit()

        # 3. Graph Convolution (AST Enrichment)
        # parent depends on child. alpha=0.8 means 0.8*parent + 0.2*child
        enriched_parent = engine.enrich_vector("parent.py", v_parent, alpha=0.8)

        check("Enrichment adds structural context (child semantics)", enriched_parent[1] > 0.1)
        check("Enrichment retains self semantics mostly", enriched_parent[0] == 0.8)

        # 4. Store Structural Drift
        delta_info = engine.store_delta("parent.py", v_parent, enriched_parent, drift_type="structural")
        check("Store structural drift success", delta_info["drift_type"] == "structural")
        check("Structural drift is tracked (> 0)", delta_info["drift_score"] > 0)

        # 5. Stats
        st = engine.stats()
        check("Stats reflect AST edges", st["packet_edges (AST)"] == 1)

    finally:
        anchor.close()

    print("=" * 50)
    print(f"  Result: {passed}/{total} PASS")
    return 0 if passed == total else 1

if __name__ == "__main__":
    args = sys.argv[1:]
    if "--self-test" in args:
        sys.exit(_run_self_test())
    elif "--stats" in args:
        print(json.dumps(SemanticDeltaEngine().stats(), indent=2, ensure_ascii=False))